# Unrealistic Alpha — Cells 01–04 (Pilot Smoke)

Goal: prove **build → tokenize → batch → forward → backward → step → checkpoint → resume** on Kaggle before any bulk work.

**YOUR MANUAL STEPS** (details in the last cell): 1) Upload this notebook to Kaggle → 2) Accelerator: GPU ON → 3) Internet ON → 4) Run All → 5) Save output version (artifacts) → 6) Report the smoke table back.

Constitution: 100B budget, Tier-B licenses, provenance per shard. These 4 cells spend ~20 min GPU, validating machinery only.

In [ ]:
# ── Cell 01: environment / reproducibility ──
!pip install -q sentencepiece datasets matplotlib
import sys, json, hashlib, random
import torch
print('python', sys.version.split()[0])
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(), '| gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — turn GPU ON (Settings → Accelerator)')
SEED = 42
random.seed(SEED); torch.manual_seed(SEED)
ALPHA_CFG = {'hidden_size': 1280, 'intermediate_size': 3456, 'num_hidden_layers': 22, 'num_attention_heads': 20, 'num_key_value_heads': 5, 'vocab_size': 48000, 'max_ctx': 2048, 'seed': SEED}
print('config hash:', hashlib.sha256(json.dumps(ALPHA_CFG, sort_keys=True).encode()).hexdigest()[:12])
assert torch.cuda.is_available(), 'STOP: enable GPU accelerator and re-run'

## Cell 02 — Tokenizer (smoke scale)

Samples EN (TinyStories) + math (OpenWebMath) + code-flavored text, trains an **8K SPM Unigram** to validate machinery. The real **48K** trains on the full corpus in later cells with the fertility hard gate. If this cell fails, nothing downstream can work — fix here, cheaply.

In [ ]:
# ── Cell 02: sample corpus → SPM smoke tokenizer ──
from datasets import load_dataset
en = load_dataset('roneneldan/TinyStories', split='train', streaming=True)
ma = load_dataset('open-web-math/open-web-math', split='train', streaming=True)
docs = []
for i, r in enumerate(en):
    docs.append(r['text'].replace('\n', ' ').strip())
    if len(docs) >= 6000: break
for i, r in enumerate(ma):
    docs.append(r['text'].replace('\n', ' ').strip()[:2000])
    if len(docs) >= 8000: break
open('/kaggle/working/smoke_corpus.txt', 'w').write('\n'.join(docs))
print('docs:', len(docs))
import sentencepiece as spm
spm.SentencePieceTrainer.train(input='/kaggle/working/smoke_corpus.txt', model_prefix='/kaggle/working/smoke_spm', vocab_size=8000, model_type='unigram', character_coverage=1.0, input_sentence_size=8000, shuffle_input_sentence=True, unk_id=0, bos_id=1, eos_id=2, pad_id=-1)
sp = spm.SentencePieceProcessor('/kaggle/working/smoke_spm.model')
print('vocab:', sp.get_piece_size())
# fertility demo (informational at smoke scale; hard gate runs on 48K later)
tests = {'english': 'The quick brown fox jumps over the lazy dog.', 'python': 'def factorial(n):\n    return 1 if n < 2 else n * factorial(n - 1)', 'math': 'The integral of x^2 dx equals x^3/3 + C.'}
print(f"{'domain':<8} {'tok/char':>8}")
for d, t in tests.items():
    print(f'{d:<8} {len(sp.encode(t)) / max(1, len(t)):>8.3f}')

## Cell 03 — Architecture smoke

Inline 50M-class proxy with **identical code paths** to the 450M (RMSNorm, GQA, RoPE-NeoX, SwiGLU, tied embeddings). Checks: param count, forward/backward shapes, memory footprint vs 16GB budget. Full 450M lives in `alpha/src/model/alpha_lm.py`.

In [ ]:
# ── Cell 03: tiny LLaMA proxy (50M class) ──
import math, torch, torch.nn as nn, torch.nn.functional as F
class RMSNorm(nn.Module):
    def __init__(self, d, e=1e-6): super().__init__(); self.w = nn.Parameter(torch.ones(d)); self.e = e
    def forward(self, x): return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.e) * self.w
def rope_cache(T, hd, th=100000.0, dev='cuda'):
    inv = 1.0 / (th ** (torch.arange(0, hd, 2, device=dev).float() / hd))
    f = torch.outer(torch.arange(T, device=dev).float(), inv)
    return torch.cos(f), torch.sin(f)
def apply_rope(x, cos, sin):
    d = x.shape[-1]; x1, x2 = x[..., :d//2], x[..., d//2:]; c, s = cos[:, :x.shape[2], :], sin[:, :x.shape[2], :]
    return torch.cat([x1*c - x2*s, x1*s + x2*c], dim=-1)
class TinyLM(nn.Module):
    def __init__(self, V=8000, h=512, ff=1376, L=8, H=8, kv=2):
        super().__init__(); self.tok = nn.Embedding(V, h)
        self.lns1 = nn.ModuleList([RMSNorm(h) for _ in range(L)]); self.lns2 = nn.ModuleList([RMSNorm(h) for _ in range(L)])
        self.qs = nn.ModuleList([nn.Linear(h, H*64, bias=False) for _ in range(L)]); self.ks = nn.ModuleList([nn.Linear(h, kv*64, bias=False) for _ in range(L)])
        self.vs = nn.ModuleList([nn.Linear(h, kv*64, bias=False) for _ in range(L)]); self.os = nn.ModuleList([nn.Linear(H*64, h, bias=False) for _ in range(L)])
        self.gs = nn.ModuleList([nn.Linear(h, ff, bias=False) for _ in range(L)]); self.us = nn.ModuleList([nn.Linear(h, ff, bias=False) for _ in range(L)])
        self.ds = nn.ModuleList([nn.Linear(ff, h, bias=False) for _ in range(L)]); self.n = RMSNorm(h); self.L, self.H, self.kv = L, H, kv
    def forward(self, ids):
        B, T = ids.shape; cos, sin = rope_cache(T, 64); cos, sin = cos.unsqueeze(0), sin.unsqueeze(0)
        x = self.tok(ids)
        for i in range(self.L):
            h = self.lns1[i](x)
            q = self.qs[i](h).view(B, T, self.H, 64).transpose(1, 2); k = self.ks[i](h).view(B, T, self.kv, 64).transpose(1, 2); v = self.vs[i](h).view(B, T, self.kv, 64).transpose(1, 2)
            q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
            k = k.repeat_interleave(self.H // self.kv, dim=1); v = v.repeat_interleave(self.H // self.kv, dim=1)
            a = F.scaled_dot_product_attention(q, k, v, is_causal=True)
            x = x + self.os[i](a.transpose(1, 2).reshape(B, T, -1))
            h2 = self.lns2[i](x); x = x + self.ds[i](F.silu(self.gs[i](h2)) * self.us[i](h2))
        return self.n(x) @ self.tok.weight.T
    def count(self): return sum(p.numel() for p in self.parameters())
m = TinyLM().cuda()
print('params:', f'{m.count()/1e6:.1f}M')
xb = torch.randint(0, 8000, (2, 128), device='cuda')
loss = torch.nn.functional.cross_entropy(m(xb)[:, :-1].reshape(-1, 8000), xb[:, 1:].reshape(-1))
loss.backward()
print('fwd+bwd OK, loss:', round(loss.item(), 3), '| mem GB:', round(torch.cuda.max_memory_allocated()/1e9, 2))
print('450M target ≈ 444M: h1280/L22/H20/kv5/ff3456/V48K/ctx2048 (see alpha_450m.json)')

## Cell 04 — Tiny end-to-end: train → checkpoint → resume

~300 steps on the smoke corpus, save, reload, verify loss continuity (the 12h-session survival skill). If resume isn't bit-continuous here, the 100B run is doomed — prove it now.

In [ ]:
# ── Cell 04: 300-step train + checkpoint + resume check ──
import sentencepiece as spm, matplotlib.pyplot as plt
sp = spm.SentencePieceProcessor('/kaggle/working/smoke_spm.model')
ids_all = []
text = open('/kaggle/working/smoke_corpus.txt').read()
ids_all = sp.encode(text)[:200000]
T, BS = 128, 4
def batch(pos):
    import random; idx = [random.randrange(0, len(ids_all) - T - 1) for _ in range(BS)]
    x = torch.tensor([ids_all[i:i+T] for i in idx], device='cuda'); y = torch.tensor([ids_all[i+1:i+T+1] for i in idx], device='cuda')
    return x, y
opt = torch.optim.AdamW(m.parameters(), lr=3e-4)
losses = []
for step in range(300):
    x, y = batch(step); opt.zero_grad()
    l = torch.nn.functional.cross_entropy(m(x)[:, :-1].reshape(-1, 8000), y[:, :-1].reshape(-1))
    l.backward(); torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
    losses.append(l.item())
    if (step+1) % 100 == 0: print(f'step {step+1} loss {l.item():.3f}', flush=True)
torch.save({'model': m.state_dict(), 'opt': opt.state_dict(), 'step': 300, 'rng': torch.get_rng_state()}, '/kaggle/working/smoke_ckpt.pt')
print('saved smoke_ckpt.pt')
# resume: fresh model + opt, load, verify continuity on fixed batch
torch.manual_seed(0); xb, yb = batch(0)
l_before = torch.nn.functional.cross_entropy(m(xb)[:, :-1].reshape(-1, 8000), yb[:, :-1].reshape(-1)).item()
m2 = TinyLM().cuda(); o2 = torch.optim.AdamW(m2.parameters(), lr=3e-4)
ck = torch.load('/kaggle/working/smoke_ckpt.pt'); m2.load_state_dict(ck['model']); o2.load_state_dict(ck['opt'])
l_after = torch.nn.functional.cross_entropy(m2(xb)[:, :-1].reshape(-1, 8000), yb[:, :-1].reshape(-1)).item()
print(f'resume check: before={l_before:.4f} after={l_after:.4f} match={abs(l_before-l_after) < 1e-4}')
assert abs(l_before - l_after) < 1e-4, 'RESUME BROKEN'
plt.plot(losses); plt.title('smoke loss'); plt.xlabel('step'); plt.savefig('/kaggle/working/smoke_loss.png'); print('plot saved')
print('SMOKE COMPLETE: machinery proven. Save output version, report table back.')

## What YOU do (manual steps)

1. **Upload**: Kaggle → Code → New Notebook → `...` menu → Upload this `.ipynb` (or paste cells).
2. **Settings**: Accelerator = **GPU** (T4x2), Internet = **ON**, Environment = latest.
3. **Run All**, watch Cell 01 assert GPU, Cell 02 build vocab, Cell 04 loss fall + resume check pass (~20 min).
4. **Save**: File → Save Version → *Save & Run All (commit)* so outputs persist; download `smoke_spm.model`, `smoke_ckpt.pt`, `smoke_loss.png` from output, or save artifacts as a Kaggle Dataset (Add Data → New Dataset).
5. **Quota**: this run spends ~0.3 GPU-hours of your 30h/week. Session cap 12h — this finishes well inside one session.
6. **Report back**: config hash + vocab size + fertility table + final loss + resume True/False.

Then: cells 05–20 (license gate → pillars → packing → training) unlock after smoke passes. Full 100B run needs funded GPUs; everything else runs free.